Evalutaing LSTM

In [ ]:
# Predict probabilities
y_probs = model.predict(X_test[:5])

# Get top predicted product for each sequence
y_pred_ids = np.argmax(y_probs, axis=1)

# Decode product names
predicted_products = product_encoder.inverse_transform(y_pred_ids)
true_products = product_encoder.inverse_transform(y_test[:5])

for i in range(5):
    print(f"Input Sequence: {product_encoder.inverse_transform(X_test[i])}")
    print(f"Predicted NBO: {predicted_products[i]}")
    print(f"Actual Next:    {true_products[i]}")
    print("—" * 40)


In [ ]:
import numpy as np

# Predict probabilities
y_pred_probs = model.predict(X_test[:1000])

# Top-K predictions manually
k = 5
top_k_preds = np.argsort(y_pred_probs, axis=1)[:, -k:]

# Count how often true label appears in top-k
y_true_subset = y_test[:1000]
correct = sum(y in preds for y, preds in zip(y_true_subset, top_k_preds))

top_k_accuracy = correct / len(y_true_subset)
print(f"Top-{k} Accuracy: {top_k_accuracy:.4f}")



In [ ]:
# Construct user-level final order test set
SEQ_LEN = 3
X_test_eval = []
y_test_eval = []

for user, seq in user_sequences.items():
    if len(seq) < SEQ_LEN + 1:
        continue
    last_seq = seq[-(SEQ_LEN + 1):]
    X_test_eval.append(last_seq[:-1])
    y_test_eval.append(last_seq[-1])

# Encode using LabelEncoder
X_test_encoded = [product_encoder.transform(seq) for seq in X_test_eval]
y_test_encoded = product_encoder.transform(y_test_eval)

# Reduced samples
X_test_final_small = X_test[:1000]
y_test_final_small = y_test[:1000]

# Now predict
y_pred_probs = model.predict(X_test_final_small, batch_size=64)
y_pred_top1 = np.argmax(y_pred_probs, axis=1)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# Basic metrics
accuracy = accuracy_score(y_test_final_small, y_pred_top1)
precision = precision_score(y_test_final_small, y_pred_top1, average='macro', zero_division=0)
recall = recall_score(y_test_final_small, y_pred_top1, average='macro', zero_division=0)
f1 = f1_score(y_test_final_small, y_pred_top1, average='macro', zero_division=0)

print(f"Top-1 Accuracy:  {accuracy:.4f}")
print(f"Precision:       {precision:.4f}")
print(f"Recall:          {recall:.4f}")
print(f"F1 Score:        {f1:.4f}")


In [ ]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from collections import Counter

MAX_TRANSACTIONS = 500_00
TOP_PRODUCTS = 1000         # Keep only top-N most frequent products

# === STEP 1: Load and Preprocess ===
products = pd.read_csv("E:/Dissertation Sem/Papers for dissertation/products.csv")
order_prior = pd.read_csv("E:/Dissertation Sem/Papers for dissertation/order_products__prior.csv")
orders = pd.read_csv("E:/Dissertation Sem/Papers for dissertation/orders.csv")

# Filter for prior orders only
orders = orders[orders['eval_set'] == 'prior']
order_data = order_prior.merge(orders[['order_id', 'user_id']], on='order_id')

# Map product_id to product_name
product_map = products.set_index('product_id')['product_name'].to_dict()
order_data['product_name'] = order_data['product_id'].map(product_map)

all_products_flat = [item for basket in transactions for item in basket]
top_items = set([item for item, _ in Counter(all_products_flat).most_common(TOP_PRODUCTS)])

filtered_txns = [
    [item for item in basket if item in top_items]
    for basket in transactions[:MAX_TRANSACTIONS]
    if any(item in top_items for item in basket)
]

print(f"Using {len(filtered_txns)} transactions with top {TOP_PRODUCTS} products")

# === STEP 2: Encode
te = TransactionEncoder()
te_matrix = te.fit(filtered_txns).transform(filtered_txns)
basket_df = pd.DataFrame(te_matrix, columns=te.columns_)

# === STEP 3: Apriori
freq_itemsets = apriori(basket_df, min_support=0.01, use_colnames=True)
rules = association_rules(freq_itemsets, metric="confidence", min_threshold=0.3)
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())

# Ensure antecedents/consequents are sets
rules['antecedents'] = rules['antecedents'].apply(lambda x: set(x))
rules['consequents'] = rules['consequents'].apply(lambda x: set(x))

# === STEP 4: Evaluation on Simulated Test Set ===
user_sequences = order_data.groupby('user_id')['product_name'].apply(list).to_dict()
SEQ_LEN = 3
test_baskets = []
actual_next_items = []

for user, seq in user_sequences.items():
    if len(seq) > SEQ_LEN:
        test_baskets.append(seq[:-1])
        actual_next_items.append(seq[-1])

# === STEP 5: Prediction Function ===
def recommend_next(basket_products, rules_df, top_n=3):
    basket_set = set(basket_products)
    matched = rules_df[rules_df['antecedents'].apply(lambda x: x.issubset(basket_set))]
    matched = matched.sort_values(by=['lift', 'confidence'], ascending=False)
    recommended = matched['consequents'].explode().value_counts().head(top_n)
    return list(recommended.index)

# === STEP 6: Evaluate Predictions ===
y_pred = []
y_true = []

for basket, true in zip(test_baskets, actual_next_items):
    preds = recommend_next(basket, rules, top_n=1)
    if preds:
        y_pred.append(preds[0])
        y_true.append(true)

# Convert to NumPy arrays
if y_pred:
    y_pred = np.array(y_pred)
    y_true = np.array(y_true)

    accuracy = np.mean(y_true == y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    print(f"Top-1 Accuracy (Apriori): {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
else:
    print("No valid predictions made. Evaluation skipped.")
